In [60]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [61]:
import pandas as pd
import numpy as np
!pip install catboost -q
from catboost import CatBoostRegressor

In [62]:
DATA_PATH = "/content/drive/MyDrive/freight_Rate_Assessment(ML)"

train_df = pd.read_csv(f"{DATA_PATH}/train-test.csv")
validation_df = pd.read_csv(f"{DATA_PATH}/validation.csv")
december_df = pd.read_csv(f"{DATA_PATH}/december-chart-inputs.csv")

print(train_df.shape)
print(validation_df.shape)
print(december_df.shape)

(48000, 14)
(12000, 13)
(31, 7)


In [38]:
##features
def prepare_features(df):
    df = df.copy()

    # Convert date
    df["date"] = pd.to_datetime(df["date"])

    # Date features
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["day_of_week"] = df["date"].dt.dayofweek

    # Safe engineered features
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    df["weight_per_distance"] = (
        df["weight"] / (df["distance"] + 1)
    )

    df["lat_diff"] = (
        df["delivery_lat"] - df["pickup_lat"]
    ).abs()

    df["lon_diff"] = (
        df["delivery_lon"] - df["pickup_lon"]
    ).abs()

    df["manhattan_dist"] = (
        df["lat_diff"] + df["lon_diff"]
    )

    return df

In [39]:
#prepare raw data
final_train = prepare_features(train_df)
final_validation = prepare_features(validation_df)


In [40]:
##
# Weight median by equipment
weight_medians = final_train.groupby("equipment")["weight"].median()
global_weight_median = final_train["weight"].median()

# Market index median by month
market_month_medians = final_train.groupby("month")["market_index"].median()
global_market_median = final_train["market_index"].median()


In [41]:
def fill_missing_values(df):
    df = df.copy()

    # Weight
    df["weight"] = df["weight"].fillna(
        df["equipment"].map(weight_medians)
    )

    df["weight"] = df["weight"].fillna(
        global_weight_median
    )

    # Market index
    df["market_index"] = df["market_index"].fillna(
        df["month"].map(market_month_medians)
    )

    df["market_index"] = df["market_index"].fillna(
        global_market_median
    )

    # Recalculate feature that depends on weight
    df["weight_per_distance"] = (
        df["weight"] / (df["distance"] + 1)
    )

    # Market / quote relationship
    df["market_quote_ratio"] = (
        df["market_index"] /
        (df["quote_signal"] + 1e-5)
    )

    return df

In [42]:
final_train = fill_missing_values(final_train)
final_validation = fill_missing_values(final_validation)


final features

In [43]:
cat_features = [
    "pickup",
    "delivery",
    "equipment"
]

In [44]:
final_features = [
    "pickup",
    "delivery",
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "equipment",
    "weight",
    "market_index",
    "quote_signal",
    "month",
    "day",
    "day_of_week",
    "is_weekend",
    "weight_per_distance",
    "lat_diff",
    "lon_diff",
    "manhattan_dist",
    "market_quote_ratio"
]

build final training data



In [45]:
X_final = final_train[final_features].copy()
y_final = final_train["posted_rate"]

X_validation = final_validation[final_features].copy()

for col in cat_features:
    X_final[col] = X_final[col].fillna("Missing").astype(str)
    X_validation[col] = X_validation[col].fillna("Missing").astype(str)

print("Training rows:", len(X_final))
print("Validation rows:", len(X_validation))


Training rows: 48000
Validation rows: 12000


Train Final CatBoost


In [46]:
final_model = CatBoostRegressor(
    iterations=669,
    depth=8,
    learning_rate=0.03,
    loss_function="MAE",
    random_seed=42,
    verbose=False
)

final_model.fit(
    X_final,
    y_final,
    cat_features=cat_features
)

print("Final model training complete.")

Final model training complete.


In [47]:
##predict validation
validation_predictions = final_model.predict(
    X_validation
)

validation_predictions[:10]

array([ 859.61335342, 4990.33359966, 5233.44278761, 3974.73461981,
       1852.88521717, 4356.07500498, 5225.5627276 , 2785.99027773,
       1073.11497425, 1077.69781984])

In [58]:
%%writefile /content/drive/MyDrive/freight_Rate_Assessment(ML)/score.py
from __future__ import annotations

import argparse
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


EXPECTED_ROWS = 12_000
EXPECTED_IDS = {
    f"TE-{index:06d}"
    for index in range(1, EXPECTED_ROWS + 1)
}

DECEMBER_DATES = pd.date_range(
    "2025-12-01",
    "2025-12-31",
    freq="D"
)

FIXED_PICKUP = "Lexington"
FIXED_DELIVERY = "Fort Wayne"
FIXED_DISTANCE = 360.0
FIXED_EQUIPMENT = "Dry Van"
FIXED_WEIGHT = 32_000.0


def fail(message: str) -> None:
    raise SystemExit(f"ERROR: {message}")


def read_csv(path: Path, label: str) -> pd.DataFrame:
    if not path.is_file():
        fail(f"{label} file not found: {path}")

    try:
        return pd.read_csv(path)
    except Exception as exc:
        fail(f"could not read {label}: {exc}")


def numeric_series(
    frame: pd.DataFrame,
    column: str,
    label: str
) -> pd.Series:

    values = pd.to_numeric(
        frame[column],
        errors="coerce"
    )

    if values.isna().any() or not np.isfinite(values).all():
        fail(f"{label} contains invalid {column} values")

    return values.astype(float)


def validate_predictions(
    predictions: pd.DataFrame
) -> None:

    if list(predictions.columns) != [
        "load_id",
        "predicted_rate"
    ]:
        fail(
            "predictions must contain exactly two columns "
            "in this order: load_id,predicted_rate"
        )

    if len(predictions) != EXPECTED_ROWS:
        fail(
            f"predictions must contain exactly "
            f"{EXPECTED_ROWS:,} rows"
        )

    if (
        predictions["load_id"].isna().any()
        or predictions["load_id"].duplicated().any()
    ):
        fail(
            "predictions contains missing or duplicate load_id values"
        )

    submitted_ids = set(
        predictions["load_id"].astype(str)
    )

    missing = EXPECTED_IDS - submitted_ids
    extra = submitted_ids - EXPECTED_IDS

    if missing or extra:
        fail(
            "prediction IDs do not match the validation set "
            f"(missing={len(missing)}, extra={len(extra)})"
        )

    predicted_rate = numeric_series(
        predictions,
        "predicted_rate",
        "predictions"
    )

    if (predicted_rate <= 0).any():
        fail(
            "predictions contains non-positive predicted_rate values"
        )


def validate_december(
    frame: pd.DataFrame
) -> pd.DataFrame:

    columns = [
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "weight",
        "date",
        "predicted_rate"
    ]

    if list(frame.columns) != columns:
        fail(
            "December predictions must keep the original "
            "seven columns and column order"
        )

    result = frame.copy()

    result["date"] = pd.to_datetime(
        result["date"],
        errors="coerce"
    )

    if result["date"].isna().any():
        fail(
            "December predictions contains invalid dates"
        )

    result["distance"] = numeric_series(
        result,
        "distance",
        "December predictions"
    )

    result["weight"] = numeric_series(
        result,
        "weight",
        "December predictions"
    )

    result["predicted_rate"] = numeric_series(
        result,
        "predicted_rate",
        "December predictions"
    )

    if result["date"].duplicated().any():
        fail(
            "December predictions contains duplicate dates"
        )

    if (
        len(result) != 31
        or set(result["date"]) != set(DECEMBER_DATES)
    ):
        fail(
            "December predictions must contain one row for "
            "every day from 2025-12-01 to 2025-12-31"
        )

    if not result["pickup"].eq(FIXED_PICKUP).all():
        fail(
            f"December pickup must be {FIXED_PICKUP} for all rows"
        )

    if not result["delivery"].eq(FIXED_DELIVERY).all():
        fail(
            f"December delivery must be {FIXED_DELIVERY} for all rows"
        )

    if not np.isclose(
        result["distance"],
        FIXED_DISTANCE
    ).all():
        fail(
            f"December distance must be {FIXED_DISTANCE:g} for all rows"
        )

    if not result["equipment"].eq(
        FIXED_EQUIPMENT
    ).all():
        fail(
            f"December equipment must be "
            f"{FIXED_EQUIPMENT} for all rows"
        )

    if not np.isclose(
        result["weight"],
        FIXED_WEIGHT
    ).all():
        fail(
            f"December weight must be {FIXED_WEIGHT:g} for all rows"
        )

    if (result["predicted_rate"] <= 0).any():
        fail(
            "December predicted_rate values must be positive"
        )

    return result.sort_values("date")


def save_december_chart(
    december: pd.DataFrame,
    output: Path
) -> None:

    figure, axis = plt.subplots(
        figsize=(10.8, 4.8),
        dpi=180
    )

    color = "#064A56"

    axis.plot(
        december["date"],
        december["predicted_rate"],
        color=color,
        linewidth=2.6,
        marker="o",
        markersize=3.2,
    )

    floor = float(
        december["predicted_rate"].min()
    )

    axis.fill_between(
        december["date"],
        december["predicted_rate"],
        floor - max(10.0, floor * 0.02),
        color=color,
        alpha=0.08,
    )

    axis.set_title(
        "Candidate: December 2025 Predicted Load Rate",
        loc="left",
        fontsize=15,
        fontweight="bold",
        pad=12
    )

    axis.set_ylabel("Predicted rate ($)")

    axis.grid(
        axis="y",
        color="#D9E2E4",
        linewidth=0.8
    )

    axis.spines[["top", "right"]].set_visible(False)

    axis.spines[
        ["left", "bottom"]
    ].set_color("#9DAFB3")

    axis.tick_params(
        axis="x",
        rotation=35
    )

    axis.text(
        0,
        -0.40,
        "Fixed inputs: Lexington to Fort Wayne | "
        "360 miles | Dry Van | 32,000 lb | only date changes",
        transform=axis.transAxes,
        fontsize=9.5,
        color="#455A60",
    )

    figure.tight_layout(
        rect=(0, 0.12, 1, 1)
    )

    figure.savefig(
        output,
        bbox_inches="tight"
    )

    plt.close(figure)


def main() -> None:

    parser = argparse.ArgumentParser(
        description=(
            "Validate candidate output files and "
            "generate the fixed December chart."
        )
    )

    parser.add_argument(
        "--predictions",
        required=True,
        help="CSV with load_id,predicted_rate"
    )

    parser.add_argument(
        "--december-predictions",
        required=True,
        help="Completed data/december_chart_inputs.csv"
    )

    parser.add_argument(
        "--output-dir",
        default="scorer_results"
    )

    args = parser.parse_args()

    validate_predictions(
        read_csv(
            Path(args.predictions),
            "predictions"
        )
    )

    december = validate_december(
        read_csv(
            Path(args.december_predictions),
            "December predictions"
        )
    )

    output = Path(args.output_dir)

    output.mkdir(
        parents=True,
        exist_ok=True
    )

    chart = output / "candidate_december.png"

    save_december_chart(
        december,
        chart
    )

    print(
        f"Validated {EXPECTED_ROWS:,} final predictions."
    )

    print(
        "Validated 31 fixed December predictions."
    )

    print(
        f"Created chart: {chart}"
    )

    print(
        "Final validation metrics are calculated "
        "by Spotter after submission."
    )


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/Assessment/score.py


Create validation_predictions.csv

In [48]:
submission = pd.DataFrame({
    "load_id": validation_df["load_id"],
    "predicted_rate": validation_predictions
})

submission.head()

,load_id,predicted_rate
0,TE-000001,859.613353
1,TE-000002,4990.333600
2,TE-000003,5233.442788
3,TE-000004,3974.734620
4,TE-000005,1852.885217


In [49]:
print("Rows:", len(submission))
print("Missing predictions:", submission["predicted_rate"].isnull().sum())
print("Duplicate load IDs:", submission["load_id"].duplicated().sum())
print("Minimum prediction:", submission["predicted_rate"].min())

Rows: 12000
Missing predictions: 0
Duplicate load IDs: 0
Minimum prediction: 192.77248354055155


In [50]:
submission.to_csv(
    f"{DATA_PATH}/validation_predictions.csv",
    index=False
)

print("validation_predictions.csv saved.")

validation_predictions.csv saved.


In [51]:
december_ready = december_df.copy()

# Convert date
december_ready["date"] = pd.to_datetime(december_ready["date"])

# Date features
december_ready["month"] = december_ready["date"].dt.month
december_ready["day"] = december_ready["date"].dt.day
december_ready["day_of_week"] = december_ready["date"].dt.dayofweek
december_ready["is_weekend"] = (december_ready["day_of_week"] >= 5).astype(int)

In [52]:
december_ready = december_df.copy()

# Date
december_ready["date"] = pd.to_datetime(december_ready["date"])
december_ready["month"] = december_ready["date"].dt.month
december_ready["day"] = december_ready["date"].dt.day
december_ready["day_of_week"] = december_ready["date"].dt.dayofweek
december_ready["is_weekend"] = (december_ready["day_of_week"] >= 5).astype(int)

# Fixed coordinates from the provided data
december_ready["pickup_lat"] = 36.99152
december_ready["pickup_lon"] = -84.99876
december_ready["delivery_lat"] = 41.31561
december_ready["delivery_lon"] = -85.36206

# Use daily market information from validation.csv
validation_temp = validation_df.copy()
validation_temp["date"] = pd.to_datetime(validation_temp["date"])

daily_market = (
    validation_temp
    .groupby("date")[["market_index", "quote_signal"]]
    .median()
    .reset_index()
)

december_ready = december_ready.merge(
    daily_market,
    on="date",
    how="left"
)

# Same engineered features used by the final model
december_ready["weight_per_distance"] = (
    december_ready["weight"] / (december_ready["distance"] + 1)
)

december_ready["lat_diff"] = (
    december_ready["delivery_lat"] - december_ready["pickup_lat"]
).abs()

december_ready["lon_diff"] = (
    december_ready["delivery_lon"] - december_ready["pickup_lon"]
).abs()

december_ready["manhattan_dist"] = (
    december_ready["lat_diff"] + december_ready["lon_diff"]
)

december_ready["market_quote_ratio"] = (
    december_ready["market_index"] /
    (december_ready["quote_signal"] + 1e-5)
)

print(december_ready.isnull().sum())

pickup                  0
delivery                0
distance                0
equipment               0
weight                  0
date                    0
predicted_rate         31
month                   0
day                     0
day_of_week             0
is_weekend              0
pickup_lat              0
pickup_lon              0
delivery_lat            0
delivery_lon            0
market_index            0
quote_signal            0
weight_per_distance     0
lat_diff                0
lon_diff                0
manhattan_dist          0
market_quote_ratio      0
dtype: int64


In [54]:
X_december = december_ready[final_features].copy()

for col in cat_features:
    X_december[col] = (
        X_december[col]
        .fillna("Missing")
        .astype(str)
    )

print("December input shape:", X_december.shape)

December input shape: (31, 20)


In [55]:
december_predictions = final_model.predict(X_december)

december_output = december_df.copy()
december_output["predicted_rate"] = december_predictions

december_output.head()

,pickup,delivery,distance,equipment,weight,date,predicted_rate
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,832.476367
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,844.274498
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,854.020825
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,856.226428
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,850.586229


In [56]:
print("Rows:", len(december_output))
print("Missing predictions:", december_output["predicted_rate"].isnull().sum())
print("Minimum prediction:", december_output["predicted_rate"].min())
print("Maximum prediction:", december_output["predicted_rate"].max())

Rows: 31
Missing predictions: 0
Minimum prediction: 832.4763669220747
Maximum prediction: 871.9389633184126


In [57]:
december_output.to_csv(
    f"{DATA_PATH}/december_predictions.csv",
    index=False
)

print("december_predictions.csv saved.")

december_predictions.csv saved.


final cheack


In [53]:
print("Validation predictions:")
print(submission.shape)

#print("\nDecember predictions:")
#print(december_output.shape)

print("\nValidation predicted rate summary:")
print(submission["predicted_rate"].describe())

Validation predictions:
(12000, 2)

Validation predicted rate summary:
count    12000.000000
mean      2360.285910
std       1360.862458
min        192.772484
25%       1269.147269
50%       2033.329533
75%       3360.206390
max       6670.569029
Name: predicted_rate, dtype: float64


In [59]:
!python "/content/drive/MyDrive/freight_Rate_Assessment(ML)/score.py" \
  --predictions "/content/drive/MyDrive/freight_Rate_Assessment(ML)/validation_predictions.csv" \
  --december-predictions "/content/drive/MyDrive/freight_Rate_Assessment(ML)/december_predictions.csv" \
  --output-dir "/content/drive/MyDrive/freight_Rate_Assessment(ML)/scorer_results"

Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: /content/drive/MyDrive/Assessment/scorer_results/candidate_december.png
Final validation metrics are calculated by Spotter after submission.
